# Reproduction of the public BilevelPSR example, and a scope statement on EME device-level insertion loss

**Author** Chan King Ho ｜ **Date** 2026-09-23
**Repository** (scripts, data, run records): **https://github.com/55093693k-dot/psr-bilevel-eme-scope**

This is an engineering record in three parts: a reproduction, one single-variable test, and the scope
statement that follows from that test. It reports what was measured, what was changed, which of the
resulting numbers may be quoted, and what this report does not solve.

The claims below are statements about **this device and this numerical route**, bounded by the
conditions in §4. They are not statements about the solver.


---

## 0. How to use this notebook

**What is free and what is paid.** Every cloud run in this notebook prints its **cost estimate
first**; nothing is submitted until you pass `--submit`. The offline analysis steps
(`analyze_coupler_eme*.py`) read an existing `.hdf5` and cost **0 FlexCredit**.

**You need your own Tidy3D API key** for the cloud runs (`pip install tidy3d==2.12`, then configure
the key the usual way). The offline analysis only needs `tidy3d` plus an `.hdf5`.

**What is runnable here.** Exactly one cell is meant to be executed: §0.1 runs offline with numpy
alone — no Tidy3D account, no API key, no network — and re-derives the two numbers this report is
built on, asserting them. Everything else is text you can read without running anything.

**Where the code comes from.** The repository's scripts are reproduced below **verbatim, as source
listings** (fenced `python` blocks, not cells):

> https://github.com/55093693k-dot/psr-bilevel-eme-scope

They are command-line programs: they receive their arguments through `sys.argv` / `argparse` and find
their neighbours through `__file__`, so **executing them as notebook cells raises**. Each listing is
headed by the exact command that runs it instead. The repository also carries the validation data,
the run records and the figure; the scripts are self-contained (geometry, materials, mesh, sources and
ports are defined in them) and need no file outside the repository.

**This notebook is a report, not a demo.** The numbers in §1–§2 come from runs that are already
complete; §0.1 lets you check the arithmetic behind them without a solver, and §4 states what may and
may not be quoted from them.

**Licence.** Code MIT; data, figures and text CC BY 4.0. If you reuse a number, please carry the scope
statement in §4 with it — see §5 for what may and may not be quoted.


---

### 0.1 Verify this report in one minute

The cell below needs only numpy. It recomputes the column sums and the conversion of §2 straight
from the two matrices in the run records, prints them next to the values quoted in the report, and
asserts that (a) the two agree to within the rounding budget of the 5-decimal records, and (b) the two
claims of §2 hold — the column sums **rise** rather than fall when port modes are added, and the
conversion moves by a few tenths of a percent. No solver, no account, no key.


In [ ]:
# -*- coding: utf-8 -*-
"""04_verify_numbers.py -- re-derive the headline table of the report, offline.

Runs with numpy only: no tidy3d, no Tidy3D account, no API key, no network.
The two matrices below are transcribed from the run records in the companion
repository (notes/eme_coupler_4modes.md and notes/eme_coupler_6modes_convergence.md,
section 2, T = |S21|^2 at lambda = 1.55 um).  Rows are output port modes, columns
are input port modes, both in EME basis order.

Because the records quote 5 decimal places, sums recomputed from them may differ
from the sums quoted in the report by up to ~2e-5 -- that is the rounding budget,
and the tolerances below are set to it.  The point of the cell is that the two
claims the report is built on can be checked without a solver:

  (a) the EME column sums do not fall when port modes are added -- they rise;
  (b) the TE1 -> lower-arm TE0 conversion changes by a few tenths of a percent.
"""
import numpy as np

np.set_printoptions(precision=5, suppress=True)

# --- 4 port modes (the row used for the first EME run) -----------------------
T4 = np.array([
    [0.99807, 0.00005, 0.00011, 0.00176],
    [0.00002, 0.99525, 0.00012, 0.00457],
    [0.00188, 0.00411, 0.09977, 0.89326],
    [0.00001, 0.00055, 0.89998, 0.09931],
])

# --- 6 port modes (only the port-mode count changed) -------------------------
T6 = np.array([
    [0.99805, 0.00005, 0.00011, 0.00128, 0.00049, 0.00001],
    [0.00002, 0.99233, 0.00010, 0.00294, 0.00242, 0.00214],
    [0.00189, 0.00447, 0.10017, 0.65299, 0.23921, 0.00116],
    [0.00001, 0.00063, 0.89545, 0.05980, 0.04406, 0.00001],
    [0.00000, 0.00052, 0.00404, 0.28206, 0.71335, 0.00001],
    [0.00001, 0.00194, 0.00012, 0.00084, 0.00042, 0.99640],
])

# --- numbers as quoted in the report and in the run records ------------------
QUOTED = {
    4: {"col": [0.99999, 0.99996, 0.99999, 0.99889],
        "row": [0.99999, 0.99996, 0.99902, 0.99986],
        "total": 3.99883, "mean": 0.99971},
    6: {"col": [0.99999, 0.99996, 0.99999, 0.99991, 0.99995, 0.99973],
        "row": [0.99999, 0.99996, 0.99989, 0.99998, 0.99998, 0.99973],
        "total": 5.99953, "mean": 0.999922},
}
ATOL_SUM = 3e-5      # rounding budget of the 5-decimal records
ATOL_TOT = 5e-5


def report_domain():
    """Compare the sums quoted in the report with the ones recomputed from the records."""
    for n, T in ((4, T4), (6, T6)):
        cs, rs = T.sum(axis=0), T.sum(axis=1)
        q = QUOTED[n]
        assert np.allclose(cs, q["col"], atol=ATOL_SUM), (n, "col sums", cs)
        assert np.allclose(rs, q["row"], atol=ATOL_SUM), (n, "row sums", rs)
        assert abs(cs.sum() - q["total"]) < ATOL_TOT, (n, "total", cs.sum())
        assert abs(cs.mean() - q["mean"]) < ATOL_TOT, (n, "mean", cs.mean())
        d = max(np.abs(cs - np.array(q["col"])).max(),
                np.abs(rs - np.array(q["row"])).max())
        print("%d port modes" % n)
        print("    column sums  quoted     %s" % np.array(q["col"]))
        print("                 recomputed %s" % cs)
        print("    row sums     quoted     %s" % np.array(q["row"]))
        print("                 recomputed %s" % rs)
        print("    max |recomputed - quoted| = %.0e   <- rounding budget of the 5-decimal records"
              % d)
        print("    column-sum min/max/mean  quoted %.5f / %.5f / %.6f   recomputed %.5f / %.5f / %.6f"
              % (min(q["col"]), max(q["col"]), q["mean"], cs.min(), cs.max(), cs.mean()))
        print()


def conversion():
    """The TE1 -> lower-arm-TE0 conversion, and its change with the mode count."""
    c4, c6 = T4[1, 1], T6[1, 1]
    assert abs(c4 - 0.99525) < 1e-6 and abs(c6 - 0.99233) < 1e-6, (c4, c6)
    print("TE1 -> lower-arm TE0: %.5f (%d modes) -> %.5f (%d modes)   %+.2f%%"
          % (c4, 4, c6, 6, 100.0 * (c6 / c4 - 1.0)))
    return c4, c6


def main():
    print("numpy %s -- offline check, no Tidy3D account, no API key, no network\n"
          % np.__version__)
    report_domain()
    print()
    c4, c6 = conversion()

    d_col = T6.sum(axis=0).mean() / T4.sum(axis=0).mean() - 1.0
    print("\nchange in the mean column sum: %+.4f%%  (upward, not downward)" % (100.0 * d_col))
    assert d_col > 0.0, "the report's claim is that the column sums rise"
    assert abs(100.0 * (c6 / c4 - 1.0) + 0.293) < 5e-3, "conversion change"
    print("all assertions passed -- the two claims of section 2 are reproduced offline.")


if __name__ == "__main__":
    main()


## 1. Reproduction

The reference design is the public **BilevelPSR** example (Flexcompute / Tidy3D notebook collection),
ported to an **SOI 220 nm + 90 nm partial-etch** stack. The **geometry is taken over unchanged** —
`t_si = 0.220 µm` and `t_pes = 0.090 µm` match this platform exactly, and the widths, lengths and gap
are the example's values. The parameter set is in `sim/run_bilevel_psr.py` and in §2 of the
repository README.

**What was adapted is the simulation setup, not the geometry**: the domains (taper y-span 3.2 µm,
coupler y-span 6.0 µm), the EME cell count (69 cells for the coupler segment), the port-mode counts
(2 / 4 / 6 across the runs), and the use of a single frequency are local choices. Each is stated
together with the result it belongs to.

The adiabatic path was verified in **two independent segments, by two different numerical methods**:

| Segment | Method and settings | Result |
|---|---|---|
| **taper** (x −6…109 µm, TM0 in) | **3D FDTD** — ModeSource → ModeMonitor at x = 105 µm, 2 modes; 12 steps/λ in-plane plus a 20 nm z-override; `run_time` 3.6 ps (≥ 2× the optical path time) | **TM0 → TE1 = 98.35%**; TM0 → TE0 = 0.0000; segment IL ≈ **0.072 dB** — an **upper bound**, the domain was clipped in y |
| **coupler** (x 100…410 µm, TE1 in) | **3D EME** — `EMEExplicitGrid`, 69 cells; ports at x = 105 / x = 405; **6 modes**; `constraint = passive`; single frequency 1.55 µm | **TE1 → lower-arm TE0 = 99.233%**; TE1 → upper arm 5e-5; TE0 → upper-arm TE0 = 99.805%; TE0 → lower-arm TE0 = 2e-5 |

Port modes were identified by **two independent evidences** — effective index *and* transverse field
centroid — and not by mode index:

| Port | mode | n_eff | transverse centroid (geometric) | identification |
|---|---|---|---|---|
| in, x = 105 | 0 / 1 | **2.7027 / 2.2480** | −0.0015 / −0.0167 | upper waveguide **TE0 / TE1** (independent FDTD: 2.7059 / 2.252) |
| out, x = 405 | 0 / 1 | 2.6057 / **2.4391** | −0.1039 / **−0.8662** (geometric −0.10 / −0.875) | upper-arm TE0 / **lower-arm TE0** |

**Result of the reproduction.** The path `TM0 → (taper) TE1 → (adiabatic coupler) lower-arm TE0` is
confirmed: each segment is verified by a method independent of the other, and the two segments meet
at a common plane (x = 105 µm) where the mode transferred by the taper is the mode launched into the
coupler.


The two scripts behind §1. `run_bilevel_psr.py` is the shared recipe (geometry,
materials, mesh, source/port definitions and the EME configuration); `run_fdtd_short.py` is the
taper-segment 3D FDTD run. `--stage modes` and `--pol tm` without `--submit` are **free**.

**`sim/run_bilevel_psr.py`** — run it as a script: `python sim/run_bilevel_psr.py --stage modes`

```python
# -*- coding: utf-8 -*-
"""run_bilevel_psr.py —— 复刻公开设计 **BilevelPSR**（taper/绝热耦合路线，C 波段）

公开来源：https://docs.flexcompute.com/projects/tidy3d/en/v2.11.0/notebooks/BilevelPSR.html
几何 / 材料 / 网格 / 源 / 端口的完整定义都写在本脚本内（见下方 P 与 make_* 函数），
设置说明见同目录 `SIMULATION_SETUP.md`，结论与引用限制见上一级 `README.md`。

阶段：
  --stage geometry   出结构图（俯视 + 横截面，含尺寸标注）—— 跑仿真前先核对几何
  --stage modes      本地 ModeSolver 在该示例的 4 个 x 位置复算模式，与该示例自带的核对表比对（含 neff>1.44 物理窗口检查）
"""
from __future__ import annotations

import argparse
import os
import sys

import gdstk
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
import tidy3d.web as web

for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8", errors="replace")
    except Exception:                                              # noqa: BLE001
        pass

HERE = os.path.dirname(os.path.abspath(__file__))

# ---------------------------------------------------------------------------
# 该示例参数（逐项与 notebook 一致）
# ---------------------------------------------------------------------------
P = dict(
    L_blt=100.0, L_ac=300.0, L_s=5.0, L_t=30.0,
    w_1=0.45, w_2=0.55, w_pes=1.55, w_3=0.85,
    w_4=0.2, w_5=0.65, w_6=0.5, gap=0.2,
    t_pes=0.09, t_si=0.22,
    R=300.0, theta=np.pi / 30.0,
    sidewall_angle=10 * np.pi / 180.0,
    inf_eff=1e5,
    lda0=1.55, band=(1.50, 1.58), n_pts=9,
)


def si_medium():
    return td.material_library["cSi"]["Palik_Lossless"]


def freqs_of(p):
    return td.C_0 / np.linspace(p["band"][0], p["band"][1], p["n_pts"])


def make_cell(p):
    """该示例的 gdstk 定义：上波导 + 下波导（逐行照抄 notebook）。"""
    inf_eff = p["inf_eff"]
    L_blt, L_ac, L_s, L_t = p["L_blt"], p["L_ac"], p["L_s"], p["L_t"]
    w_1, w_2, w_3, w_5 = p["w_1"], p["w_2"], p["w_3"], p["w_5"]
    w_4, w_6, gap = p["w_4"], p["w_6"], p["gap"]
    R, theta = p["R"], p["theta"]

    cell = gdstk.Cell("device")
    top_wg = gdstk.RobustPath((-inf_eff, 0), w_1, layer=1, datatype=0)
    top_wg.horizontal(0)
    top_wg.horizontal(L_blt / 2, w_2)
    top_wg.horizontal(L_blt, w_3)
    top_wg.horizontal(L_blt + L_s)
    top_wg.segment((L_blt + L_s + L_ac, (w_5 - w_3) / 2), w_5)
    top_wg.horizontal(L_blt + L_s + L_ac + L_t)
    top_wg.arc(R, -np.pi / 2, -np.pi / 2 + theta)
    top_wg.arc(R, np.pi / 2 + theta, np.pi / 2)
    top_wg.horizontal(inf_eff)
    cell.add(top_wg)

    bottom_wg = gdstk.RobustPath((L_blt + L_s, (-w_4 - w_3 - 2 * gap) / 2), w_4,
                                 layer=1, datatype=0)
    bottom_wg.segment((L_blt + L_s + L_ac, (-w_3 - 2 * gap - w_6) / 2), w_6)
    bottom_wg.arc(R, np.pi / 2, np.pi / 2 - theta)
    bottom_wg.arc(R, -np.pi / 2 - theta, -np.pi / 2)
    bottom_wg.horizontal(inf_eff)
    cell.add(bottom_wg)
    return cell


def make_structures(p):
    si = si_medium()
    v = [(0, p["w_1"] / 2), (p["L_blt"] / 2, p["w_pes"] / 2), (p["L_blt"], p["w_3"] / 2),
         (p["L_blt"], -p["w_3"] / 2), (p["L_blt"] / 2, -p["w_pes"] / 2), (0, -p["w_1"] / 2)]
    pes = td.Structure(
        geometry=td.PolySlab(vertices=v, axis=2, slab_bounds=(0, p["t_pes"]),
                             sidewall_angle=p["sidewall_angle"]),
        medium=si, name="partially_etched_slab")
    geos = td.PolySlab.from_gds(make_cell(p), gds_layer=1, axis=2,
                                 slab_bounds=(0, p["t_si"]),
                                 sidewall_angle=p["sidewall_angle"])
    wgs = [td.Structure(geometry=g, medium=si, name=f"wg_{i}") for i, g in enumerate(geos)]
    return [pes] + wgs


# ---------------------------------------------------------------------------
# 结构图
# ---------------------------------------------------------------------------
def draw_xsec_mid(ax, p):
    """x=L_blt/2 截面：部分刻蚀平板（w_pes @ t_pes）+ 上波导（w_2 @ t_si）。"""
    ax.add_patch(plt.Rectangle((-1.6, -0.5), 3.2, 1.1, fc="#cfe3f5", ec="none", zorder=0))
    ax.add_patch(plt.Rectangle((-p["w_pes"] / 2, 0), p["w_pes"], p["t_pes"],
                               fc="#e6a17b", ec="k", lw=0.5, zorder=1))
    ax.add_patch(plt.Rectangle((-p["w_2"] / 2, 0), p["w_2"], p["t_si"],
                               fc="#c94f2b", ec="k", lw=0.6, zorder=2))
    dim_h(ax, -p["w_pes"] / 2, p["w_pes"] / 2, p["t_si"] + 0.12,
          f"w_pes={p['w_pes']*1000:.0f}nm (partial etch slab)")
    dim_h(ax, -p["w_2"] / 2, p["w_2"] / 2, -0.20, f"w_2={p['w_2']*1000:.0f}nm")
    dim_v(ax, 0, p["t_si"], 1.30, f"t_si={p['t_si']*1000:.0f}nm", dx=-0.35)
    dim_v(ax, 0, p["t_pes"], 1.05, f"t_pes={p['t_pes']*1000:.0f}nm", dx=-0.30)
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-0.55, 0.8)
    ax.set_xlabel("y (µm)")
    ax.set_ylabel("z (µm)")
    ax.set_title(f"Cross-section @ x=L_blt/2 = {p['L_blt']/2:.0f} µm")


def draw_xsec_coupler(ax, p):
    """x=L_blt+L_s+L_ac 截面：上波导 w_5（中心 y=(w_5-w_3)/2）+ 下波导 w_6，间隙 gap。"""
    y_top = (p["w_5"] - p["w_3"]) / 2
    y_bot = (-p["w_3"] - 2 * p["gap"] - p["w_6"]) / 2
    ax.add_patch(plt.Rectangle((-2.0, -1.6), 4.0, 2.0, fc="#cfe3f5", ec="none", zorder=0))
    ax.add_patch(plt.Rectangle((y_top - p["w_5"] / 2, 0), p["w_5"], p["t_si"],
                               fc="#c94f2b", ec="k", lw=0.6, zorder=2))
    ax.add_patch(plt.Rectangle((y_bot - p["w_6"] / 2, 0), p["w_6"], p["t_si"],
                               fc="#c94f2b", ec="k", lw=0.6, zorder=2))
    dim_h(ax, y_top - p["w_5"] / 2, y_top + p["w_5"] / 2, p["t_si"] + 0.10,
          f"w_5={p['w_5']*1000:.0f}nm (top)")
    dim_h(ax, y_bot - p["w_6"] / 2, y_bot + p["w_6"] / 2, -0.18, f"w_6={p['w_6']*1000:.0f}nm (bottom)")
    dim_h(ax, y_top - p["w_5"] / 2, y_bot + p["w_6"] / 2, 0.42,
          f"gap={p['gap']*1000:.0f}nm  (edge-to-edge)", dy=0.02)
    dim_v(ax, 0, p["t_si"], -1.85, f"t_si={p['t_si']*1000:.0f}nm", dx=-0.25)
    ax.set_xlim(-2.0, 2.0)
    ax.set_ylim(-1.6, 0.9)
    ax.set_xlabel("y (µm)")
    ax.set_ylabel("z (µm)")
    ax.set_title(f"Cross-section @ end of adiabatic coupler (x={p['L_blt']+p['L_s']+p['L_ac']:.0f} µm)")


def dim_h(ax, x0, x1, y, text, fs=7.5, dy=0.0):
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle="<->", mutation_scale=8,
                                 color="#333333", lw=1.0, shrinkA=0, shrinkB=0))
    ax.text((x0 + x1) / 2, y + dy, text, ha="center", va="bottom", fontsize=fs)


def dim_v(ax, y0, y1, x, text, fs=7.5, dx=0.0):
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch((x, y0), (x, y1), arrowstyle="<->", mutation_scale=8,
                                 color="#333333", lw=1.0, shrinkA=0, shrinkB=0))
    ax.text(x + dx, (y0 + y1) / 2, text, ha="left", va="center", fontsize=fs)


def small_sim(p, x_max=None, y_span=7.0, z_span=1.75):
    """用于绘图/模式求解的**局部**仿真（全长 435 µm 的全域不在此处构建：见 NOTES 的 EME 说明）。"""
    si = si_medium()
    _, sio2 = None, td.material_library["SiO2"]["Palik_Lossless"]
    x_max = x_max if x_max is not None else (p["L_blt"] + p["L_s"] + p["L_ac"] + p["L_t"] + 5)
    return td.Simulation(
        center=((x_max - 5) / 2, 0, 0.11), size=(x_max + 5, y_span, z_span),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=p["lda0"]),
        structures=make_structures(p), run_time=1e-12,
        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()), medium=sio2)


def full_sim(p):
    """**整器件**仿真域（仅用于结构绘图；不构建网格，故 ~500 µm 也无成本）。
    y 跨度取 **±6 µm**（12 µm），避免边缘波导被截断。"""
    sio2 = td.material_library["SiO2"]["Palik_Lossless"]
    x_end = p["L_blt"] + p["L_s"] + p["L_ac"] + p["L_t"] + 2 * p["R"] * p["theta"] + 20
    return td.Simulation(
        center=(x_end / 2 - 5, 0, 0.11), size=(x_end + 10, 12.0, 1.75),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=p["lda0"]),
        structures=make_structures(p), run_time=1e-12,
        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()), medium=sio2), x_end


def geometry_bounds(p):
    """打印结构总包围盒（用于确认 y 向是否被截断）。"""
    from tidy3d import GeometryGroup
    geos = [s.geometry for s in make_structures(p)]
    try:
        b = GeometryGroup(geometries=geos).bounds
        print("structure bounds (xmin,xmax,ymin,ymax,zmin,zmax) =",
              tuple(round(v, 3) for v in (b[0][0], b[1][0], b[0][1], b[1][1], b[0][2], b[1][2])))
    except Exception as e:                                          # noqa: BLE001
        print("bounds failed:", repr(e))


def stage_geometry(p):
    """三张图：① 整器件俯视（两层）② 分段放大俯视 ③ 横截面（带尺寸）。"""
    figs = os.path.join(HERE, "figs")
    os.makedirs(figs, exist_ok=True)
    sim, x_end = full_sim(p)
    x_lo, x_hi = -5.0, x_end
    geometry_bounds(p)          # 打印结构包围盒，核对 y 域 ±6 µm 是否完整覆盖

    # ① 整器件俯视（x 与 y 比例不同，图注已说明）
    fig, axs = plt.subplots(2, 1, figsize=(24, 7), dpi=130)
    for ax, z, tag in ((axs[0], p["t_pes"] / 2, "partial-etch slab layer"),
                       (axs[1], p["t_si"] / 2, "silicon waveguide layer")):
        sim.plot(z=z, ax=ax)
        ax.set_aspect("auto")
        ax.set_xlim(x_lo, x_hi)
        ax.set_title(f"WHOLE DEVICE top view — z-slice at z={z:.3f} µm ({tag}). "
                     f"NOTE: single Si layer (t_si=220 nm) + 90-nm partial etch; "
                     f"this is a SLICE, not a second thickness. "
                     f"Total length ≈ {x_hi - x_lo:.0f} µm (x:y aspect not to scale)")
    f1 = os.path.join(figs, "bilevel_psr_topview_FULL.png")
    fig.tight_layout()
    fig.savefig(f1, bbox_inches="tight")
    plt.close(fig)

    # ② 分段放大俯视（同一层，4 段覆盖整器件）
    segs = [(x_lo, 110, "1) input + bi-level taper (w_1=0.45 → w_2=0.55 → w_3=0.85 µm)"),
            (100, 215, "2) straight (L_s=5) + adiabatic coupler entry (bottom wg w_4=0.2 µm)"),
            (205, 410, "3) adiabatic coupler (upper w_5=0.65 / lower w_6=0.5 / gap=0.2 µm)"),
            (400, x_hi, "4) output S-bends (R=300 µm) and output waveguides")]
    fig, axs = plt.subplots(len(segs), 1, figsize=(16, 16), dpi=130)
    for ax, (x0, x1, name) in zip(axs, segs):
        sim.plot(z=p["t_si"] / 2, ax=ax)
        ax.set_xlim(x0, x1)
        ax.set_aspect("auto")
        ax.set_title(name, fontsize=10)
    f2 = os.path.join(figs, "bilevel_psr_topview_ZOOM.png")
    fig.tight_layout()
    fig.savefig(f2, bbox_inches="tight")
    plt.close(fig)

    # ③ 横截面（带尺寸标注）
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi=130)
    draw_xsec_mid(axs[0], p)
    draw_xsec_coupler(axs[1], p)
    f3 = os.path.join(figs, "bilevel_psr_xsec.png")
    fig.tight_layout()
    fig.savefig(f3, bbox_inches="tight")
    plt.close(fig)

    for f in (f1, f2, f3):
        print("figure ->", f, os.path.getsize(f), "bytes")
    return f1, f2, f3


# ---------------------------------------------------------------------------
# 本地模式复核（免费）：在该示例的 4 个 x 位置复算模式，与该示例自带的核对表比对
# ---------------------------------------------------------------------------
def xsec_model(p, x):
    """按该示例参数给出 x 处的**局部截面**（宽度随 x 线性/分段插值）。"""
    y = {}
    if x <= p["L_blt"]:
        f = x / (p["L_blt"] / 2) if x <= p["L_blt"] / 2 else 1 + (x - p["L_blt"] / 2) / (p["L_blt"] / 2)
        y["w_slab"] = (p["w_1"] + (p["w_pes"] - p["w_1"]) * f) if x <= p["L_blt"] / 2 \
            else (p["w_pes"] + (p["w_3"] - p["w_pes"]) * (f - 1))
    if x <= p["L_blt"] / 2:
        y["w_top"] = p["w_1"] + (p["w_2"] - p["w_1"]) * (x / (p["L_blt"] / 2))
    elif x <= p["L_blt"]:
        y["w_top"] = p["w_2"] + (p["w_3"] - p["w_2"]) * ((x - p["L_blt"] / 2) / (p["L_blt"] / 2))
    elif x <= p["L_blt"] + p["L_s"]:
        y["w_top"] = p["w_3"]
    else:
        f = (x - (p["L_blt"] + p["L_s"])) / p["L_ac"]
        y["w_top"] = p["w_3"] + (p["w_5"] - p["w_3"]) * f
        y["y_top"] = (p["w_5"] - p["w_3"]) / 2 * f
    if x >= p["L_blt"] + p["L_s"]:
        f = (x - (p["L_blt"] + p["L_s"])) / p["L_ac"]
        y["w_bot"] = p["w_4"] + (p["w_6"] - p["w_4"]) * f
        y["y_bot"] = ((-p["w_4"] - p["w_3"] - 2 * p["gap"]) / 2
                      + ((-p["w_3"] - 2 * p["gap"] - p["w_6"]) / 2
                         - (-p["w_4"] - p["w_3"] - 2 * p["gap"]) / 2) * f)
    return y


def xsec_sim_plane(p, x, y_c=0.0):
    """**Tidy3D 约定**：器件沿 x 传播 → 模式平面为 **x-normal**（面内轴 = y(横向分离), z(高度)）。
    局部截面按该示例的局部尺寸构建；仿真域 y 取 **±6 µm**，平面取 ±1.5 µm。"""
    si = si_medium()
    sio2 = td.material_library["SiO2"]["Palik_Lossless"]
    m = xsec_model(p, x)
    structs = []
    if "w_slab" in m:
        structs.append(td.Structure(geometry=td.Box(center=(0, y_c, p["t_pes"] / 2),
                                                    size=(3, m["w_slab"], p["t_pes"])), medium=si))
    structs.append(td.Structure(geometry=td.Box(center=(0, y_c + m.get("y_top", 0.0), p["t_si"] / 2),
                                                size=(3, m["w_top"], p["t_si"])), medium=si))
    if "w_bot" in m:
        structs.append(td.Structure(geometry=td.Box(center=(0, y_c + m["y_bot"], p["t_si"] / 2),
                                                    size=(3, m["w_bot"], p["t_si"])), medium=si))
    sim = td.Simulation(center=(0, 0, 0), size=(3.0, 12.0, 1.75),
                        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=p["lda0"]),
                        structures=structs, run_time=1e-13,
                        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()), medium=sio2)
    plane = td.Box(center=(0, 0, p["t_si"] / 2), size=(0.0, 3.0, 1.4))   # x-normal（Tidy3D 约定）
    return sim, plane


def stage_modes(p):
    """该示例自带的核对表：x=0 → TE0/TM0；L_blt/2 → TE0/混合；L_blt+L_s → TE0/TE1；耦合器末端 → 上下波导 TE0。"""
    positions = [0.0, p["L_blt"] / 2, p["L_blt"] + p["L_s"],
                 p["L_blt"] + p["L_s"] + p["L_ac"]]
    from tidy3d.plugins.mode import ModeSolver
    for x in positions:
        sim, plane = xsec_sim_plane(p, x)
        spec = td.ModeSpec(num_modes=4, target_neff=2.6).updated_copy(num_pml=(6, 6))
        data = ModeSolver(simulation=sim, plane=plane, mode_spec=spec,
                          freqs=[td.C_0 / p["lda0"]]).solve()
        neff = np.asarray(data.n_eff.values).ravel()
        pf = data.pol_fraction
        te = tm = None
        try:
            te = np.asarray(pf.sel(pol="te").values).ravel()
            tm = np.asarray(pf.sel(pol="tm").values).ravel()
        except Exception as e:                                     # noqa: BLE001
            print(f"  !! pol_fraction 读取失败（易错项）：{e!r}")
            print(f"     dims={getattr(pf, 'dims', None)}  coords={list(getattr(pf, 'coords', {}))}")
            arr = np.asarray(pf.values)
            if arr.ndim >= 2 and arr.shape[-1] == 2:               # 兜底：最后一轴 = (te, tm)
                te, tm = arr[..., 0].ravel(), arr[..., 1].ravel()
                print("     → 已用兜底方式（最后一轴=(te,tm)）读取；请核对该假设")
            else:
                print("     → 兜底失败：**本次模式识别仅按 neff，偏振过滤未生效**（必须在此处停下修正）")
        print(f"--- x = {x:.1f} µm ---")
        for i, n in enumerate(neff):
            flag = "OK" if n > 1.44 else "!! neff<n_clad(1.44) → 泄漏模（易错项）"
            extra = "" if te is None else f"  te={te[i]:.3f} tm={tm[i]:.3f}"
            print(f"  mode{i}: neff={n:.4f}{extra}   {flag}")


def eme_explicit_grid(p, num_modes=4):
    """第二次修正（**唯一单一变量 = EME 网格**；其余设置与上一轮完全一致）。

    依据 Tidy3D API（docs.flexcompute.com, v2.9.3/v2.11.2）：
      `EMEExplicitGrid(mode_specs=[每格一个], boundaries=[内部边界; 个数 = 格数-1; 严格递增])`
    分区（x 从 -10 µm 起）：
      ① bi-level taper 0 → L_blt：2 µm/格（profile 缓变）
      ② 突变界面（L_blt−5 → L_blt+L_s+5）：0.5 µm/格（90 nm 台阶 + 下波导骤现 w_4=0.2 µm）
      ③ 绝热耦合器 L_blt+L_s → x_dev：5 µm/格
    """
    x_dev = p["L_blt"] + p["L_s"] + p["L_ac"] + p["L_t"] + 2 * p["R"] * p["theta"]
    x_start, x_end_sim = -10.0, x_dev + 10.0
    x_tap = p["L_blt"]
    x_junc = p["L_blt"] + p["L_s"]

    xs = []
    xs += list(np.arange(0.0, x_tap + 1e-6, 2.0))                     # ① 2 µm
    xs += list(np.arange(x_tap - 5.0, x_junc + 5.0 + 1e-6, 0.5))      # ② 0.5 µm
    xs += list(np.arange(x_junc, x_dev + 1e-6, 5.0))                  # ③ 5 µm
    bnd = sorted({round(float(v), 5) for v in xs
                  if x_start + 1e-6 < float(v) < x_end_sim - 1e-6})   # 只保留内部边界
    specs = [td.EMEModeSpec(num_modes=num_modes) for _ in range(len(bnd) + 1)]
    print(f"EMEExplicitGrid: {len(bnd) + 1} cells  ({len(bnd)} internal boundaries) "
          f"| taper 2 µm / junction 0.5 µm / coupler 5 µm")
    return td.EMEExplicitGrid(mode_specs=specs, boundaries=tuple(bnd))


def make_eme_sim(p, num_cells=60, num_modes=4):
    """Tidy3D EME 配方：axis=0（x 向传播）+ EME 网格 + port_offsets + 系数监视器。
    此后 eme_grid_spec 改为 `EMEExplicitGrid`（在几何突变处分界），不再是 EMEUniformGrid(60)。"""
    sio2 = td.material_library["SiO2"]["Palik_Lossless"]
    x_dev = p["L_blt"] + p["L_s"] + p["L_ac"] + p["L_t"] + 2 * p["R"] * p["theta"]
    pad = 20.0
    return td.EMESimulation(
        center=(x_dev / 2, 0, 0.11), size=(x_dev + 2 * pad, 12.0, 1.75),   # y=±6 µm
        medium=sio2, structures=make_structures(p),
        axis=0, freqs=freqs_of(p),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=p["lda0"]),
        eme_grid_spec=eme_explicit_grid(p, num_modes=num_modes),   # 显式网格（在几何突变处分界）
        port_offsets=(pad / 2, pad / 2),
        constraint="unitary",          # Tidy3D 建议：抑制不稳定辐射模
        store_port_modes=True,         # 关键诊断：保存端口模式（用于判定 mode_index ↔ 波导）
        monitors=[td.EMEModeSolverMonitor(name="port_modes", num_modes=num_modes,
                                         size=(0, td.inf, td.inf), center=(x_dev / 2, 0, 0.11)),
                  td.EMEModeSolverMonitor(name="modes_in", num_modes=num_modes,
                                          size=(0, td.inf, td.inf), center=(-pad / 2, 0, 0.11)),
                  td.EMEModeSolverMonitor(name="modes_out", num_modes=num_modes,
                                          size=(0, td.inf, td.inf), center=(x_dev + pad / 2, 0, 0.11)),
                  td.EMECoefficientMonitor(name="coeffs", size=(td.inf, td.inf, td.inf))],
    )


def stage_eme(p, submit=False):
    """EME 验证：输入 TE0/TM0 注入 → 读输出端两个波导的 TE0/TM0 透射（判 TM→TE>90%）。"""
    sim = make_eme_sim(p, num_modes=2)   # 单一变量：端口模式 4→2（去掉近截止/频率跳变模）
    print("=== 几何核对报告 ===")
    print(" 传播方向 = +x | 分离方向 = y（两波导横向分开）| 高度方向 = z（Si 220nm + 90nm 部分刻蚀，单层）")
    print(f" 端口平面 = **x-normal**（垂直于 +x）| 仿真域 size={sim.size} center={sim.center}")
    print(f" 模式索引：输入 mode_index_in 0=TE0 / 1=TM0（按 neff 识别）；输出端按 neff 区分上下波导")
    print("=== 提交前校验 ===")
    sim.validate_pre_upload()
    print(" validate_pre_upload: PASS")
    task = f"bilevel_psr_eme_Lac{p['L_ac']:.0f}"
    job = web.Job(simulation=sim, task_name=task, verbose=False)
    print(" estimate_cost (FlexCredit):", web.estimate_cost(job.task_id))
    if not submit:
        print(" [dry-run] 未提交；加 --submit 才提交（会先打印成本估算）。")
        return None
    data = job.run(path=os.path.join(HERE, f"data_{task}.hdf5"))
    print(" === smatrix 结构 ===")
    try:
        S = data.smatrix
        print(" smatrix dims:", list(getattr(S, "dims", {})))
        print(" S21 dims:", list(getattr(S.S21, "dims", {})), " shape:", getattr(S.S21, "shape", None))
        print(" coords:", {k: (list(v.values)[:6] if hasattr(v, "values") else v)
                           for k, v in getattr(S.S21, "coords", {}).items()})
    except Exception as e:                                          # noqa: BLE001
        print(" smatrix 读取失败：", repr(e))
    try:
        pm = data["port_modes"]
        print(" port_modes neff (看能否区分上下波导):", np.asarray(pm.n_eff.values))
    except Exception as e:                                          # noqa: BLE001
        print(" port_modes 读取失败：", repr(e))
    print(" data file:", os.path.join(HERE, f"data_{task}.hdf5"))
    return data


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="geometry", choices=["geometry", "modes", "eme"])
    ap.add_argument("--submit", action="store_true", help="EME：真正提交（默认 dry-run 只报成本）")
    a = ap.parse_args()
    print("BilevelPSR 复刻参数：", {k: round(v, 4) for k, v in P.items()
                                    if isinstance(v, float)})
    if a.stage == "geometry":
        stage_geometry(P)
    elif a.stage == "modes":
        stage_modes(P)
    else:
        stage_eme(P, submit=a.submit)


if __name__ == "__main__":
    main()
```


**`sim/run_fdtd_short.py`** — run it as a script: `python sim/run_fdtd_short.py --pol tm`

```python
# run_fdtd_short.py -- 公开示例的做法（3D FDTD + ModeSource + ModeMonitor）的【缩短版】决定性实验
# 目的：只验证 bi-level taper 段（x = 0…105 µm）的 **TM0 -> TE1** 转换（该示例机理的表征点）
#       若该转换成立 ⇒ 器件机理在本平台成立，EME 的坏结果 = 方法（有限模式基）问题
# 用法：python run_fdtd_short.py            -> dry-run（只报成本，0 FlexCredit）
#       python run_fdtd_short.py --submit   -> 提交（浅 batch：TE + TM 两个源）
import os
import sys

import numpy as np
import tidy3d as td
import tidy3d.web as web

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import run_bilevel_psr as rbp

P = rbp.P
LAM = 1.55
X_IN = -4.0                     # 输入模式源位置（位于 0.45 µm 输入直波导内）
X_MON = P["L_blt"] + P["L_s"]   # = 105 µm：该示例机理的表征点（此处 mode0=TE0, mode1=TE1）
X0, X1 = -6.0, 109.0            # 仿真域 x 范围（含 taper 100 + 直段 5 + 余量）
Y_SPAN, Z_SPAN = 3.2, 2.2       # y 3.2：taper 横向 ±0.83 外留 >λ/2 的 PML 余量（易错项）

freq = td.C_0 / LAM
# z 向保留已验证的 20 nm 分辨率（避免"粗网格出结论"）；面内用 37 nm 降本
_ov = td.MeshOverrideStructure(
    geometry=td.Box(center=((X0 + X1) / 2, 0.0, 0.11), size=(X1 - X0, Y_SPAN, 1.0)),
    dl=(0.037, 0.037, 0.02),
)
common = dict(
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=12, wavelength=LAM, override_structures=(_ov,)),
    structures=rbp.make_structures(P),
    boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()),
    medium=td.material_library["SiO2"]["Palik_Lossless"],
    run_time=3.6e-12,          # 约束：L=109 µm, ng≈4.5 -> t_transit≈1.6 ps，run_time ≥ 2×
)
mon = td.ModeMonitor(center=(X_MON, 0.0, 0.11), size=(0.0, Y_SPAN, Z_SPAN),
                     freqs=[freq], mode_spec=td.ModeSpec(num_modes=2), name="taper_out")


def build(idx):
    src = td.ModeSource(
        center=(X_IN, 0.0, 0.11), size=(0.0, Y_SPAN, Z_SPAN), direction="+",
        source_time=td.GaussianPulse(freq0=freq, fwidth=freq / 12),
        mode_spec=td.ModeSpec(num_modes=2), mode_index=idx,
    )
    return td.Simulation(center=((X0 + X1) / 2, 0.0, 0.0), size=(X1 - X0, Y_SPAN, Z_SPAN),
                         sources=[src], monitors=[mon], **common)


s0 = build(0)     # TE0 注入
s0.validate_pre_upload()
s1 = build(1)     # TM0 注入
s1.validate_pre_upload()
print("=== shortened FDTD: bi-level taper only (x = %.0f..%.0f um) ===" % (X0, X1))
print("mode source x=%.1f | mode monitor x=%.1f | grid=20 steps/lambda | all-side PML" % (X_IN, X_MON))
print("mesh cells (approx):", int(np.prod([c / 0.02 for c in s1.grid_spec.grid_size(s1)]) if False else 0) or "n/a")

job_te = web.Job(simulation=s0, task_name="psr_taper_short_TE", verbose=False)
job_tm = web.Job(simulation=s1, task_name="psr_taper_short_TM", verbose=False)
POL = sys.argv[sys.argv.index("--pol") + 1] if "--pol" in sys.argv else "tm"   # tm | te | both
jobs = {}
if POL in ("te", "both"):
    jobs["TE"] = job_te
if POL in ("tm", "both"):
    jobs["TM"] = job_tm
print("polarization:", POL, "| jobs:", list(jobs))
tot = 0.0
for k, j in jobs.items():
    c = web.estimate_cost(j.task_id)
    tot += c
    print("estimate %s (FlexCredit): %s" % (k, c))
print("estimate TOTAL (FlexCredit): %s" % tot)

if "--submit" not in sys.argv:
    print("[dry-run] not submitted")
    sys.exit(0)
for k, j in jobs.items():
    res = j.run(path=os.path.join(HERE, "data_taper_%s.hdf5" % k))
    amp = res["taper_out"].amps.sel(direction="+")
    T = (np.abs(np.asarray(amp.values).squeeze()) ** 2)
    print("%s input  -> T(mode0)=%.4f  T(mode1)=%.4f" % (k, T.ravel()[0], T.ravel()[1]))
```


## 2. Single-variable test

**Purpose.** To obtain a device-level **absolute** insertion loss from the EME route.

**Starting point.** The coupler-segment EME run with **4 port modes** gives S-matrix column sums of
**0.99889–0.99999** — there is no loss channel inside the port-mode basis — while the device function
in the same run reads 99.525%.

A second reason not to take that 99.525% at face value: the same device, run on this route, gives
results that differ by **three orders of magnitude** depending on the constraint and the port-mode
count used.

| run | constraint | port modes | result |
|---|---|---|---|
| all-device | `unitary` | 4 | conversion **92.4–96.7%** across the C-band (9 frequencies) |
| all-device | `passive` | 2 | **TE0 → out0 = 0.14%** (IL **28.539 dB**); TM0 → out1 = 0.03% (IL **34.981 dB**) |

Both rows describe the same device on the same route. They cannot both be device performance — what
changed between them is the constraint and the port-mode count. Neither is quotable as device
performance, and neither is used as such anywhere in this report.

Provenance of the two rows: the `passive` row is reproducible from this repository
(`sim/run_eme_passive.py`, analysis output in `notes/log_an_passive_m2.txt`); the `unitary` row is
read from that run's record, which is not shipped here.

Rather than argue about which of them to believe, the port-mode basis itself was put under test — the
single-variable test below.

**Pre-registered plan** (written before the run — the question and its criterion; not a prediction of
the outcome):

- **question**: do the column sums drop when port modes are added — i.e. does a real loss channel
  appear inside the port-mode basis?
- **criterion**: the key channel (TE1 → lower-arm TE0) must change by **< 2%** between 4 and 6 port
  modes, and any decrease in the column sums must be accounted for.

**Test configuration — one variable changed:**

| held fixed | changed |
|---|---|
| domain (x 100…410 µm, y 6.0 µm, z 1.75 µm); port positions (x = 105 / 405); EME grid (`EMEExplicitGrid`, 69 cells); frequency (single, 1.55 µm); `constraint = passive` | port-mode count **4 → 6** |

**Result:**

| quantity (λ = 1.55 µm) | 4 modes | 6 modes | Δ |
|---|---|---|---|
| TE1 → lower-arm TE0 (device function) | 0.99525 | **0.99233** | −0.00292 (−0.29%) |
| TE0 → upper-arm TE0 (through) | 0.99807 | 0.99805 | −0.00002 |
| TE1 → upper-arm TE0 | 0.00005 | 0.00005 | 0 |
| TE0 → lower-arm TE0 | 0.00002 | 0.00002 | 0 |
| **column sums, min…max** | 0.99889 … 0.99999 | **0.99973 … 0.99999** | **the minimum moves up** |
| mean column sum | 0.999707 | 0.999922 | up |

**Outcome.** The question was put to measurement, and the answer is **no**: the minimum column sum
moved from 0.99889 to 0.99973 — **upward**, not across 1. Adding port modes did not open a
radiation-loss channel in this structure.

The criterion was met at the same time (Δ = −0.29% < 2%): the conversion value **converged** while the
loss channel did not appear. Two facts that have to be reported together, because either one read
alone is misleading.

![Column sums and conversion versus port mode count](attachment:colsum_and_conversion_vs_modes.png)

*Figure. Left: the per-input column sums at 4 and 6 port modes, against the 1.0000 line. Right: the
conversion at the same two points. Both axes are port-mode-basis quantities — the right panel is not
a device efficiency. The figure is an external (matplotlib) re-plot of the tabulated results, not a
solver export.*

---


The coupler-segment runs and their offline analysis. `--modes 4` produced the
4-mode row and `--modes 6` the 6-mode row; between them **only the port-mode count changed**. The
analysers are **offline** (0 FlexCredit) and take the `.hdf5` path as an argument.

**`sim/run_eme_coupler.py`** — run it as a script: `python sim/run_eme_coupler.py --modes 6 --y 6`

```python
# run_eme_coupler.py -- 耦合器段（绝热段）EME 验证
# 域 x 100…410 µm；端口严格放在该示例的耦合器边界 x=105（入口，TE1 注入）/ x=405（出口）
# 单频 λ=1.55 µm + constraint="passive"（要绝对 IL，不能 unitary）+ 存端口模式（按 neff 认模）
# 用法：python run_eme_coupler.py [--modes 4] [--y 6] [--submit]   （不带 --submit = dry-run，0 FlexCredit）
import os
import sys

import numpy as np
import tidy3d as td
from tidy3d import web

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import run_bilevel_psr as rbp                              # 复用已验证几何/EME 配方

P = rbp.P
LAM = 1.55
NMODES = int(sys.argv[sys.argv.index("--modes") + 1]) if "--modes" in sys.argv else 4
YS = float(sys.argv[sys.argv.index("--y") + 1]) if "--y" in sys.argv else 6.0

XP1 = P["L_blt"] + P["L_s"]                                # 105 µm：junction / 耦合器入口（该示例：mode1=TE1）
XP2 = XP1 + P["L_ac"]                                      # 405 µm：耦合器出口（该示例：mode3→底部 TE0）
X0, X1 = XP1 - 5.0, XP2 + 5.0                              # 域 100…410 µm
PAD = 5.0

# EME 网格（照抄已验证分区法）：junction 附近 0.5 µm/格 + 耦合器 5 µm/格（绝热）
# 注意（源码 eme/simulation.py:807）：EME 网格区间 = 域**扣除 port_offsets**，
#   端口面（105/405）自动成为首/末边界 ⇒ 用户 boundaries 必须**严格落在两端口之间**（易错项）。
xs = np.concatenate([np.arange(XP1 + 0.5, XP1 + PAD + 1e-9, 0.5),
                     np.arange(XP1 + PAD, XP2 + 1e-9, 5.0)])
bnd = sorted({round(float(v), 5) for v in xs if XP1 + 1e-6 < float(v) < XP2 - 1e-6})
grid = td.EMEExplicitGrid(mode_specs=[td.EMEModeSpec(num_modes=NMODES) for _ in range(len(bnd) + 1)],
                          boundaries=tuple(bnd))
assert all(abs(v - XP1) > 1e-6 and abs(v - XP2) > 1e-6 for v in bnd), \
    "端口面由 port_offsets 隐式给定，不得作为显式 boundary（否则 ValidationError）"

sim0 = rbp.make_eme_sim(P, num_modes=NMODES)               # 已验证配方 → 只改域/网格/约束/频率/监视器
sim = sim0.updated_copy(
    center=((X0 + X1) / 2, 0.0, 0.11), size=(X1 - X0, YS, 1.75),
    eme_grid_spec=grid, constraint="passive", freqs=[td.C_0 / LAM],
    port_offsets=(PAD, PAD),
    monitors=[td.EMEModeSolverMonitor(name="modes_in", num_modes=NMODES,
                                      size=(0, td.inf, td.inf), center=(XP1, 0.0, 0.11)),
              td.EMEModeSolverMonitor(name="modes_out", num_modes=NMODES,
                                      size=(0, td.inf, td.inf), center=(XP2, 0.0, 0.11)),
              td.EMECoefficientMonitor(name="coeffs", size=(td.inf, td.inf, td.inf))],
)
sim.validate_pre_upload()
print("=== coupler EME: 域 x %.0f…%.0f µm | 端口 x=%.0f(入口) / %.0f(出口) ===" % (X0, X1, XP1, XP2))
print("modes/port=%d | EME cells=%d (%d internal bnd; 0.5µm@junction + 5µm@coupler) | y=%.1f | f=%d | constraint=%s"
      % (NMODES, len(bnd) + 1, len(bnd), YS, len(sim.freqs), sim.constraint))

task = "psr_coupler_eme_m%d_y%g" % (NMODES, YS)
job = web.Job(simulation=sim, task_name=task, verbose=False)
print("estimate_cost (FlexCredit):", web.estimate_cost(job.task_id))
if "--submit" not in sys.argv:
    print("[dry-run] not submitted")
    sys.exit(0)

data = job.run(path=os.path.join(HERE, "data_%s.hdf5" % task))
S = data.smatrix.S21
print("S21 dims:", tuple(S.dims), "shape:", S.shape)
for nm, tag in (("modes_in", "入口 x=105"), ("modes_out", "出口 x=405")):
    try:
        md = getattr(data, nm)
        print("neff %s (%s):" % (nm, tag), np.round(np.real(np.atleast_2d(md.n_complex.values)[0]), 4))
    except Exception as e:                                  # noqa: BLE001
        print("neff %s: 读取失败 %s" % (nm, e))
print("|S21|^2 (out x in) =")
print(np.round(np.abs(S.values) ** 2, 4))
print("data file:", os.path.join(HERE, "data_%s.hdf5" % task))
```


**`sim/analyze_coupler_eme_modes.py`** — run it as a script: `python sim/analyze_coupler_eme_modes.py --hdf5 sim/data_psr_coupler_eme_m6_y6.hdf5`

```python
# -*- coding: utf-8 -*-
"""analyze_coupler_eme_modes.py -- 离线分析任意模式数的"耦合器段"EME 结果（0 FlexCredit）

用法：python analyze_coupler_eme_modes.py --hdf5 <data.hdf5> [--log <out.txt>]

认模规则：**按 neff + 横向场重心**自动定位关键通道，**绝不按模式序号**。
（同目录 `analyze_coupler_eme.py` 是把它写死为 4 模式/固定文件名的版本；本脚本参数化，
供 4 / 6 / 8 / … 模式复用 —— §4 那张 6 模式表就是用本脚本制表的。）

**复用前提（两点，换器件必看）**
1. 本脚本**完全离线**：只用 `tidy3d` 的 `EMESimulationData` 读 hdf5，**不调用 `web`** ⇒
   不需要 API key。运行前提是自备 `tidy3d` 与那一次运行的 `.hdf5`。
2. 下面几处是**本器件标定值**，换器件必须改：
   - `TE1 neff ≈ 2.248`（入口 x = 105 µm 的上波导 TE1）；
   - 出口两臂的**横向场重心** `−0.875`（下臂）/ `−0.10`（上臂）。
   这两个重心来自本器件的几何（w_6 = 0.50 µm / w_5 = 0.65 µm 对应的 y 位置），
   换几何必须重新计算，否则关键通道会被认错。
"""
import argparse
import os
import traceback

import numpy as np
import tidy3d as td

AP = argparse.ArgumentParser()
AP.add_argument("--hdf5", required=True)
AP.add_argument("--log", default=None)
ARGS = AP.parse_args()
H5 = os.path.abspath(ARGS.hdf5)
LOG = os.path.abspath(ARGS.log) if ARGS.log else os.path.splitext(H5)[0] + "_analysis.txt"

_LINES = []


def P(s=""):
    print(s, flush=True)
    _LINES.append(str(s))


def flush():
    with open(LOG, "w", encoding="utf-8") as fh:
        fh.write("\n".join(_LINES) + "\n")


def ycenter(d, comp):
    """每个模式的 |comp|^2 在 y 方向的**重心**（判定上/下波导）。"""
    c = getattr(d, comp, None)
    if c is None or "y" not in c.dims:
        return None
    w = np.abs(np.asarray(c.values)) ** 2
    keep = [i for i, dn in enumerate(c.dims) if dn in ("mode_index", "y")]
    prof = w.sum(axis=tuple(i for i in range(w.ndim) if i not in keep))
    if [c.dims[i] for i in keep][0] == "y":          # 不假设轴序，按维度名判定（易错项）
        prof = np.swapaxes(prof, -1, -2)             # -> (mode, y)
    y = np.asarray(c.coords["y"].values)
    tot = prof.sum(axis=1)
    tot[tot == 0] = 1.0
    return (prof * y[None, :]).sum(axis=1) / tot


def port_table(md, name):
    """打印单个端口的模式表（neff + 场能量占比 + 横向重心），返回 neff 数组。"""
    if name not in md:
        P("[%s] MISSING in monitor_data" % name)
        return None
    d = md[name]
    have = [a for a in ("n_complex", "Ex", "Ey", "Ez") if getattr(d, a, None) is not None]
    P("[%s] available: %s" % (name, have))
    nc = d.n_complex
    P("[%s] n_complex dims=%s shape=%s" % (name, nc.dims, nc.shape))
    neff = np.real(np.asarray(nc.values)).reshape(-1)
    P("[%s] neff = %s" % (name, np.round(neff, 4)))

    frac = {}
    for comp in ("Ey", "Ez", "Ex"):
        c = getattr(d, comp, None)
        if c is None:
            continue
        axes = tuple(i for i, dn in enumerate(c.dims) if dn in ("y", "z"))
        if not axes:
            continue
        frac[comp] = np.abs(np.asarray(c.values)) ** 2        # (f, mode, y, z)
        frac[comp] = frac[comp].sum(axis=axes).reshape(-1)   # -> (f*mode,)
    if frac:
        tot = np.zeros_like(frac[list(frac)[0]], dtype=float)
        for k in frac:
            tot = tot + frac[k]
        tot[tot == 0] = 1.0
        cy = ycenter(d, "Ey")
        cz = ycenter(d, "Ez")
        P("[%s] field energy fractions + transverse center of mass (per mode):" % name)
        for i in range(len(tot)):
            row = "  ".join("%s=%.3f" % (k, frac[k][i] / tot[i]) for k in frac)
            P("   mode %d: %s   neff=%.4f  y_c(Ey)=%s  y_c(Ez)=%s"
              % (i, row, neff[i],
                 ("%.4f" % cy[i]) if cy is not None else "n/a",
                 ("%.4f" % cz[i]) if cz is not None else "n/a"))
    return neff


try:
    P("hdf5: %s" % H5)
    P("exists: %s" % os.path.exists(H5))
    data = None
    for meth in ("from_hdf5", "from_file"):
        f = getattr(td.EMESimulationData, meth, None)
        if f is None:
            continue
        try:
            data = f(H5)
            P("loaded via EMESimulationData.%s()" % meth)
            break
        except Exception as e:                                  # noqa: BLE001
            P("%s failed: %r" % (meth, str(e)[:200]))
    if data is None:
        P("candidates: %s" % [a for a in dir(td.EMESimulationData) if "load" in a or "from" in a])
        flush()
        raise SystemExit(1)

    md = data.monitor_data
    P("monitor_data keys: %s" % list(md.keys()))

    P("=" * 70)
    neff_in = port_table(md, "modes_in")       # x=105 入口
    P("=" * 70)
    neff_out = port_table(md, "modes_out")     # x=405 出口

    P("=" * 70)
    S = data.smatrix
    S21 = S.S21
    P("S21 dims=%s shape=%s" % (S21.dims, S21.shape))
    s = S21
    if "sweep_index" in s.dims:
        s = s.isel(sweep_index=0)
        P("used isel(sweep_index=0)")
    else:
        P("no sweep_index dim -> used as-is (单频点；易错项：不用 squeeze)")
    order = [d for d in ("f", "mode_index_out", "mode_index_in") if d in s.dims]
    s = s.transpose(*order)
    P("after transpose: dims=%s shape=%s" % (s.dims, s.shape))
    T = np.abs(np.asarray(s.values)) ** 2                      # (nf, out, in)
    n_out, n_in = T.shape[1], T.shape[2]
    P("T shape (f,out,in) = %s ; modes/port: in=%d out=%d" % (T.shape, n_in, n_out))
    P("T = |S21|^2  (rows = mode_index_out, cols = mode_index_in):")
    with np.printoptions(precision=5, suppress=False):
        P(str(np.round(T, 5)))
    P("column sums (per INPUT mode, over %d out modes): %s" % (n_out, np.round(T.sum(axis=1), 5)))
    P("row sums    (per OUTPUT mode, over %d in modes): %s" % (n_in, np.round(T.sum(axis=2), 5)))
    P("grand total = %.6f | mean per input column = %.6f" % (T.sum(), T.sum() / n_in))

    P("=" * 70)
    t = T[0]
    try:
        tin = np.real(np.asarray(neff_in)).reshape(-1)
        tout = np.real(np.asarray(neff_out)).reshape(-1)
        P("input port modes  : %s" % ", ".join("m%d=%.4f" % (i, v) for i, v in enumerate(tin)))
        P("output port modes : %s" % ", ".join("m%d=%.4f" % (i, v) for i, v in enumerate(tout)))
        P("expect @x=105 (local FDE/FDTD): TE0~2.71, TE1~2.25")
        cy_out = ycenter(md["modes_out"], "Ey")
        i_te1 = int(np.argmin(np.abs(tin - 2.248)))
        o_bot = int(np.argmin(np.abs(np.asarray(cy_out) - (-0.875)))) if cy_out is not None else None
        o_top = int(np.argmin(np.abs(np.asarray(cy_out) - (-0.10)))) if cy_out is not None else None
        P("IDENT BY NEFF/CENTROID: i_TE1=%d (neff=%.4f) | o_bottom=%s | o_top=%s"
          % (i_te1, tin[i_te1], o_bot, o_top))
        if o_bot is not None and o_top is not None:
            P("KEY  in%d (TE1) -> out%d (bottom TE0) = %.5f   <== device function"
              % (i_te1, o_bot, t[o_bot, i_te1]))
            P("     in%d (TE1) -> out%d (top TE0)    = %.5f   (unwanted)"
              % (i_te1, o_top, t[o_top, i_te1]))
            P("     column sum of in%d = %.5f  (<=1 ; ==1.0000 means no radiation loss inside the basis)"
              % (i_te1, t[:, i_te1].sum()))
    except Exception as e:                                      # noqa: BLE001
        P("ident failed: %r" % (e,))

    flush()
    P("[done] log written to %s" % LOG)
    flush()
except Exception:                                              # noqa: BLE001
    P("TRACEBACK:")
    P(traceback.format_exc())
    flush()
    raise
```


**`sim/analyze_coupler_eme.py`** — run it as a script: `python sim/analyze_coupler_eme.py`

```python
# -*- coding: utf-8 -*-
"""analyze_coupler_eme.py -- 离线分析"耦合器段"EME 结果（0 FlexCredit）

目的（补上 run_eme_coupler.py 缺的那一步）：
  1) 从 hdf5 读**端口模式解**（EMEModeSolverMonitor @ x=105 / x=405）→ neff + 场极性
     → **按 neff / 极性认模**（不按位置猜模式身份）；
  2) 打印带标签的 T = |S21|^2（out x in）+ 逐入列和 / 逐出行和；
  3) 输出日志同时落盘（utf-8），避免控制台解码干扰（易错项）。

先用 `python analyze_coupler_eme.py` 跑（不需要 --submit，完全离线）。
"""
import os
import sys
import traceback

import numpy as np
import tidy3d as td

HERE = os.path.dirname(os.path.abspath(__file__))
H5 = os.path.join(HERE, "data_psr_coupler_eme_m4_y6.hdf5")
LOG = os.path.join(HERE, "log_an_cpl.txt")

_LINES = []


def P(s=""):
    print(s)
    _LINES.append(str(s))


def flush():
    with open(LOG, "w", encoding="utf-8") as fh:
        fh.write("\n".join(_LINES) + "\n")


def ycenter(d, comp):
    """每个模式的 |comp|^2 在 y 方向的**重心**（用于判定模式属于上/下波导）。"""
    c = getattr(d, comp, None)
    if c is None or "y" not in c.dims:
        return None
    w = np.abs(np.asarray(c.values)) ** 2
    keep = [i for i, dn in enumerate(c.dims) if dn in ("mode_index", "y")]
    prof = w.sum(axis=tuple(i for i in range(w.ndim) if i not in keep))
    if [c.dims[i] for i in keep][0] == "y":          # 不假设轴序，按维度名判定（易错项）
        prof = np.swapaxes(prof, -1, -2)             # -> (mode, y)
    y = np.asarray(c.coords["y"].values)
    tot = prof.sum(axis=1)
    tot[tot == 0] = 1.0
    return (prof * y[None, :]).sum(axis=1) / tot


def port_table(md, name):
    """打印单个端口的模式表（neff + Ey/Ez/Ex 能量占比），返回 neff 数组。"""
    if name not in md:
        P("[%s] MISSING in monitor_data" % name)
        return None
    d = md[name]
    have = [a for a in ("n_complex", "Ex", "Ey", "Ez") if getattr(d, a, None) is not None]
    P("[%s] available: %s" % (name, have))
    nc = d.n_complex
    P("[%s] n_complex dims=%s shape=%s" % (name, nc.dims, nc.shape))
    neff = np.real(np.asarray(nc.values)).reshape(-1)
    P("[%s] neff = %s" % (name, np.round(neff, 4)))

    frac = {}
    for comp in ("Ey", "Ez", "Ex"):
        c = getattr(d, comp, None)
        if c is None:
            continue
        axes = tuple(i for i, dn in enumerate(c.dims) if dn in ("y", "z"))
        if not axes:
            continue
        frac[comp] = np.abs(np.asarray(c.values)) ** 2        # (f, mode, y, z) 形态
        frac[comp] = frac[comp].sum(axis=axes).reshape(-1)   # -> (f*mode,)
    if frac:
        tot = np.zeros_like(frac[list(frac)[0]], dtype=float)
        for k in frac:
            tot = tot + frac[k]
        tot[tot == 0] = 1.0
        P("[%s] field energy fractions + transverse center of mass (per mode):" % name)
        for i in range(len(tot)):
            row = "  ".join("%s=%.3f" % (k, frac[k][i] / tot[i]) for k in frac)
            cy = ycenter(d, "Ey")
            cz = ycenter(d, "Ez")
            P("   mode %d: %s   neff=%.4f  y_c(Ey)=%s  y_c(Ez)=%s"
              % (i, row, neff[i],
                 ("%.4f" % cy[i]) if cy is not None else "n/a",
                 ("%.4f" % cz[i]) if cz is not None else "n/a"))
    return neff


try:
    P("hdf5: %s" % H5)
    P("exists: %s" % os.path.exists(H5))
    data = None
    for meth in ("from_hdf5", "from_file"):
        f = getattr(td.EMESimulationData, meth, None)
        if f is None:
            continue
        try:
            data = f(H5)
            P("loaded via EMESimulationData.%s()" % meth)
            break
        except Exception as e:                                  # noqa: BLE001
            P("%s failed: %r" % (meth, str(e)[:200]))
    if data is None:
        P("candidates: %s" % [a for a in dir(td.EMESimulationData) if "load" in a or "from" in a])
        flush()
        sys.exit(1)

    md = data.monitor_data
    P("monitor_data keys: %s" % list(md.keys()))

    P("=" * 70)
    neff_in = port_table(md, "modes_in")       # x=105 入口
    P("=" * 70)
    neff_out = port_table(md, "modes_out")     # x=405 出口

    P("=" * 70)
    S = data.smatrix
    S21 = S.S21
    P("S21 dims=%s shape=%s" % (S21.dims, S21.shape))
    order = ("f", "mode_index_out", "mode_index_in")
    # 注意：单频点（f 长度 1）时 squeeze() 会把 f 也删掉 → 用 isel 只去掉 sweep 轴（易错项）
    S21c = S21.isel(sweep_index=0).transpose(*order)
    P("after isel/transpose: dims=%s shape=%s" % (S21c.dims, S21c.shape))
    T = np.abs(np.asarray(S21c.values)) ** 2                   # (nf, out, in)
    lam = td.C_0 / np.asarray(S21c.coords["f"].values).ravel()  # µm
    P("lambda (um): %s" % np.round(lam, 4))
    P("T = |S21|^2  (rows = mode_index_out, cols = mode_index_in):")
    with np.printoptions(precision=5, suppress=False):
        P(str(np.round(T, 5)))
    P("column sums (per INPUT mode, summed over 4 out modes): %s" % np.round(T.sum(axis=1), 5))
    P("row sums    (per OUTPUT mode, summed over 4 in modes): %s" % np.round(T.sum(axis=2), 5))
    P("grand total = %.6f   mean per input column = %.6f"
      % (T.sum(), T.sum() / T.shape[2]))
    if T.shape[0] == 1:
        t = T[0]
        P("KEY conversions (single freq):")
        P("  in0 -> out0 (top TE0  -> top TE0)      = %.5f" % t[0, 0])
        P("  in1 -> out1 (TE1      -> bottom TE0)   = %.5f   <== device function" % t[1, 1])
        P("  in1 -> out0 (TE1      -> top TE0)      = %.5f   (unwanted)" % t[0, 1])
        P("  in0 -> out1 (top TE0  -> bottom TE0)   = %.5f   (crosstalk)" % t[1, 0])

    # 认模（只在拿到端口 neff 时做）
    if neff_in is not None and neff_out is not None:
        P("=" * 70)
        tin = np.real(neff_in).reshape(-1)
        tout = np.real(neff_out).reshape(-1)
        P("input port modes  : %s" % ", ".join("m%d=%.4f" % (i, v) for i, v in enumerate(tin)))
        P("output port modes : %s" % ", ".join("m%d=%.4f" % (i, v) for i, v in enumerate(tout)))
        P("expect @x=105 (from local FDE/FDTD): TE0~2.71, TE1~2.25")
        P("expect @x=405 (from the earlier full-device EME run @1.58um): top-TE0 2.6486, bottom-TE0 2.4976")
    flush()
    P("[done] log written to %s" % LOG)
    flush()
except Exception:                                              # noqa: BLE001
    P("TRACEBACK:")
    P(traceback.format_exc())
    flush()
    raise
```


The `passive` row of §2 comes from this variant, with the offline analysis output
recorded below it.

**`sim/run_eme_passive.py`** — run it as a script: `python sim/run_eme_passive.py --modes 2`

```python
# run_eme_passive.py -- 最小付费变体：EME(3D) constraint="passive" + 单波长 1.55 um
# 目的：检查含辐射/回反射损耗的**绝对插损**能否由 EME 给出
#       （`unitary` 会强制 S 矩阵酉性，把基内列和压到 1，看不到真实损耗通道）
# 用法：python run_eme_passive.py [--modes 2]  -> dry-run（0 FlexCredit，只报成本）
#       python run_eme_passive.py [--modes 2] --submit   -> 真正提交
import os
import sys

import tidy3d as td
from tidy3d import web

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import run_bilevel_psr as rbp                     # 复用已验证的几何/EME 配方

P = rbp.P
LAM = 1.55                                        # um
nmodes = int(sys.argv[sys.argv.index("--modes") + 1]) if "--modes" in sys.argv else 2
sim = rbp.make_eme_sim(P, num_modes=nmodes)
sim = sim.updated_copy(constraint="passive", freqs=[td.C_0 / LAM])
sim.validate_pre_upload()
print("=== variant: passive + num_modes=%d + single wavelength %.2f um ===" % (nmodes, LAM))
print("constraint =", sim.constraint, "| n_freqs =", len(sim.freqs),
      "| eme cells =", sim.eme_grid_spec.boundaries.size + 1)

task = "bilevel_psr_eme_passive_m%d_lam1550" % nmodes
job = web.Job(simulation=sim, task_name=task, verbose=False)
print("estimate_cost (FlexCredit):", web.estimate_cost(job.task_id))
if "--submit" not in sys.argv:
    print("[dry-run] not submitted")
    sys.exit(0)
data = job.run(path=os.path.join(HERE, "data_%s.hdf5" % task))
S = data.smatrix.S21
print("S21 dims:", tuple(S.dims), "shape:", S.shape)
print("data file:", os.path.join(HERE, "data_%s.hdf5" % task))
```


Recorded offline analysis output of the `passive`, 2-port-mode all-device run
(`notes/log_an_passive_m2.txt` in the repository) — the evidence for the `passive` row of §2.

```text
# 离线分析输出（节选）—— 全器件 EME，约束 passive，2 端口模式，单频 1.55 µm
# 产生方式：sim/run_eme_passive.py --modes 2 --submit 提交后，用同一离线分析脚本
#           读该次运行的 .hdf5，打印 |S21|^2 与逐列和（离线，0 FlexCredit）
# 用途：报告 §2 里 "TE0 -> out0 = 0.14%（IL 28.539 dB）" 的来源
file: data_bilevel_psr_eme_passive_m2_lam1550.hdf5 | lambda_um: [1.55]
|S21|^2 (rows=out, cols=in):
 [[0.0014  0.     ]
 [0.      0.00032]]
colsum (per input mode): [0.0014  0.00032]
max|S21|^2 = 0.00140
[modes_in] n_complex = [2.3502+0.j 1.7428+0.j]
[modes_out] n_complex = [2.6039+0.j 2.4418+0.j]
TE0  (in0): T(out0)=0.0014  total=0.0014  loss=0.9986 -> IL=28.539 dB
TM0  (in1): T(out1)=0.0003  total=0.0003  loss=0.9997 -> IL=34.981 dB
crosstalk: in0->out1 = 0.000000 | in1->out0 = 0.000000
```


The port-mode field map used as evidence for the mode identification in §1 is
produced offline from the same EME data.

**`sim/plot_port_mode_fields.py`** — run it as a script: `python sim/plot_port_mode_fields.py sim/data_psr_coupler_eme_m6_y6.hdf5`

```python
# -*- coding: utf-8 -*-
"""plot_port_mode_fields.py -- 离线画「耦合器段端口模式真实场分布」（0 FlexCredit）

输入：run_eme_coupler.py 已下载的 hdf5（默认取项目目录里的 6 模式跑）
输出：../data/PORT_MODE_FIELDS.png  —— 入口(x=105um)/出口(x=405um) 各模式 |Ey|^2 面图 + neff

坐标（本 EME 设置）：传播 = x；横向（两臂分开方向）= y；垂直 = z。
本脚本按**维度名**选取（不假设轴序），自动裁到模式有场的横向范围以利观察。

用法:  python plot_port_mode_fields.py [hdf5 路径]
依赖:  tidy3d, numpy, matplotlib（本地免费；不提交任何任务）
"""
import os
import sys

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import tidy3d as td  # noqa: E402

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_H5 = os.path.join(HERE, "..", "..", "..", "..", "PSR", "bilevel_psr",
                          "data_psr_coupler_eme_m6_y6.hdf5")
OUT = os.path.join(HERE, "..", "data", "PORT_MODE_FIELDS.png")


def load(path):
    for meth in ("from_hdf5", "from_file"):
        f = getattr(td.EMESimulationData, meth, None)
        if f is None:
            continue
        try:
            d = f(path)
            print("loaded via EMESimulationData.%s()" % meth)
            return d
        except Exception as e:  # noqa: BLE001
            print("%s failed: %r" % (meth, str(e)[:200]))
    return None


def mode_plane(ds, comp="Ey"):
    """返回 (w[mode, z, y], y 轴, z 轴, neff)。按维度名选取，去掉所有长度 1 的轴。"""
    c = getattr(ds, comp, None)
    if c is None:
        return None
    for d in list(c.dims):
        if c.sizes[d] == 1 and d != "mode_index":
            c = c.isel({d: 0})
    w = np.abs(np.asarray(c.values)) ** 2
    dims = list(c.dims)
    if w.ndim == 2:                       # 单模式：补一个 mode 轴
        w = w[None, :, :]
        dims = ["mode_index"] + dims
    order = [dims.index(a) for a in ("mode_index", "z", "y")]
    w = np.transpose(w, order)            # -> (mode, z(vertical), y(lateral))
    neff = np.real(np.asarray(ds.n_complex.values)).reshape(-1)
    return w, np.asarray(c.coords["y"].values), np.asarray(c.coords["z"].values), neff


def main():
    path = os.path.abspath(sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5)
    print("hdf5 exists: %s -> %s" % (os.path.exists(path), path))
    if not os.path.exists(path):
        sys.exit(1)
    data = load(path)
    if data is None:
        sys.exit(1)
    md = data.monitor_data
    print("monitor_data keys: %s" % list(md.keys()))

    keys = [k for k in ("modes_in", "modes_out") if k in md]
    panels = [(k, mode_plane(md[k])) for k in keys]
    panels = [(k, p) for k, p in panels if p is not None]
    if not panels:
        print("no usable panels")
        sys.exit(1)
    for k, p in panels:
        print("[%s] |Ey|^2 shape=%s neff=%s" % (k, p[0].shape, np.round(p[3], 4)))

    # 横向裁剪：所有模式合计有场（>1% 峰值）的 y 范围 + 20% 余量
    y = panels[0][1][1]
    prof = np.zeros_like(y)
    for _, (w, yy, zz, neff) in panels:
        prof = prof + w.sum(axis=(0, 1))
    keep = np.where(prof > 0.01 * prof.max())[0]
    margin = max(2, int(0.20 * (keep[-1] - keep[0] + 1)))
    i0 = max(0, keep[0] - margin)
    i1 = min(len(y), keep[-1] + 1 + margin)
    print("lateral crop: y = %.3f .. %.3f um (full %.3f .. %.3f)"
          % (y[i0], y[i1 - 1], y[0], y[-1]))

    nmax = max(p[0].shape[0] for _, p in panels)
    fig, axs = plt.subplots(len(panels), nmax,
                            figsize=(2.5 * nmax + 0.8, 3.0 * len(panels) + 1.0), squeeze=False)
    for r, (key, (w, yy, zz, neff)) in enumerate(panels):
        for c in range(nmax):
            ax = axs[r][c]
            if c < w.shape[0]:
                im = ax.imshow(w[c][:, i0:i1], origin="lower", aspect="auto", cmap="inferno",
                               extent=[float(yy[i0]), float(yy[i1 - 1]), float(zz[0]), float(zz[-1])])
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                ax.set_title("%s m%d  neff=%.4f" % (key, c, neff[c]), fontsize=8)
                ax.set_xlabel("lateral y (um)", fontsize=7)
            else:
                ax.axis("off")
            if c == 0:
                ax.set_ylabel("vertical z (um)", fontsize=7)
            ax.tick_params(labelsize=6)
    fig.suptitle("Port mode fields |Ey|^2 @1.55um -- adiabatic coupler section\n"
                 "(in: x=105um, out: x=405um; 6 modes, passive)", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(OUT, dpi=130)
    print("saved: %s (%d B)" % (os.path.abspath(OUT), os.path.getsize(OUT)))


if __name__ == "__main__":
    main()
```


## 3. Conclusion

Within the conditions listed in §4:

1. **The EME port-power column sums do not authorize a device-level absolute insertion loss for this
   device.** After the test in §2, this route is closed for that metric. The remaining options are an
   all-device 3D FDTD (estimated ≈ 30 FlexCredit for this device) or measurement.
2. **The two conversions are port-mode-basis conversions**, and they are referenceable as such:
   `TM0 → TE1 = 98.35%` (3D FDTD) and `TE1 → lower-arm TE0 = 99.233%` (6 modes, converged).
   They are not device efficiencies.
3. The −0.29% change is accounted for by the newly added modes. The test therefore demonstrates
   convergence **within the <2% criterion used here**; it does not demonstrate convergence to <1%.


## 4. Scope

The statement above is bounded in three layers. All three apply together.

| layer | what it bounds |
|---|---|
| **Structure** | **Long adiabatic devices** of this class — bi-level taper plus adiabatic coupler, total length ≈ 538 µm. No claim is made for short devices, interferometric or resonant devices, or structures whose port-mode basis is already large. |
| **Test conditions** | SOI 220 nm + 90 nm partial etch; single frequency λ = 1.55 µm; `constraint = passive`; port-mode counts 4 and 6; `EMEExplicitGrid` of 69 cells; domain and port positions as in §2. |
| **Conclusion** | A statement about **one route under one set of conditions**: the port-mode basis of this EME configuration cannot produce the device-level absolute insertion loss. It is **not** a statement that the solver cannot produce insertion loss, and **not** a statement about EME in general. |

**Not solved by this report:**

- the cause of the `unitary` / `passive` discrepancy at small port-mode counts (the 92.4–96.7% vs
  ≈0.14% observation in §2);
- whether a larger port-mode basis (10 or more) would change the column sums;
- the insertion loss of the **output section** (S-bend + `L_t`, x 405…537.8 µm), which was not
  simulated;
- any device-level return loss.


## 5. Referenceable values and quoting rules

**Referenceable**

- coupler segment **TE1 → lower-arm TE0 = 99.233%** — 6 modes / `passive` / single frequency
  λ = 1.55 µm; passed the 4→6 port-mode convergence check, Δ = −0.29%;
- taper segment **TM0 → TE1 = 98.35%** — 3D FDTD, λ = 1.55 µm.

**Not referenceable (uncertified)**

- **device-level absolute insertion loss / return loss.** Basis, with evidence: EME port column sums
  ≈ 1 (4 modes 0.99889–0.99999; 6 modes 0.99973–0.99999) ⇒ increasing the number of port modes does
  not expose radiation loss (falsified by measurement, 2026-09-12). That figure must come from an
  all-device 3D FDTD (estimated ≈ 30 FlexCredit here) or from measurement. In addition, the output
  section (S-bend + `L_t`, x 405…537.8 µm) was not measured.

**Single frequency only.** The values above were verified only at λ = 1.55 µm. The bandwidth curves
remain unconverged data and may be read as trends only, not as figures of merit.

**Taper-segment IL is an upper bound.** The domain was clipped in y, so radiation outside the window
is absorbed by the PML; the deviation can only be pessimistic.

> The authoritative wording of these rules is the Chinese original in the repository
> (`notes/` and the device README). The text above is a faithful translation of it — same claims,
> same scope.


## 6. Scripts and data

Everything needed to rerun the simulations is in the repository: **https://github.com/55093693k-dot/psr-bilevel-eme-scope**.
Geometry, materials, mesh, sources and ports are defined in the scripts themselves; no file outside
the repository is required.

A **Jupyter notebook version of this report** is in the same repository
(`psr_bilevel_eme_scope.ipynb`), identical to the copy submitted to the Tidy3D community examples. Its
**§0.1 cell is meant to be run** and needs `numpy` only: it re-derives §2's column sums and conversion
straight from the run records, prints them next to the values quoted here and asserts them — no Tidy3D
account, no API key, no network. The scripts appear in it as **source listings**, not as cells, because
they are command-line programs (`__file__` / `sys.argv`) and do not run as notebook cells.

```bash
pip install tidy3d==2.12                     # and configure your own API key
python sim/run_bilevel_psr.py --stage geometry          # structure figures (local, free)
python sim/run_bilevel_psr.py --stage modes             # local mode self-check (free)
python sim/run_bilevel_psr.py --stage eme               # validate + print the cost estimate
python sim/run_eme_coupler.py --modes 6 --y 6           # coupler segment (dry-run first)
python sim/run_eme_passive.py   --modes 2               # all-device, constraint=passive (dry-run first)
python sim/run_fdtd_short.py  --pol tm                  # taper segment, 3D FDTD (prints T(mode0), T(mode1) at x = 105 um)
python sim/analyze_coupler_eme.py                       # offline: mode-ID + column sums (4-mode input path)
python sim/analyze_coupler_eme_modes.py --hdf5 <hdf5>   # offline: same, any mode count (6-mode table in §2)
```

| run | configuration | cost | basis |
|---|---|---|---|
| taper segment 3D FDTD | 2-mode monitor, y-span 3.2 um | **1.7551 FlexCredit** | run record |
| all-device EME | 4 modes / 157 cells | **1.5725 FlexCredit** | run record |
| all-device EME | 2 modes / 157 cells | **1.335 FlexCredit** | run record |
| all-device EME | `passive`, 2 modes | **0.7727 FlexCredit** | run record |
| coupler segment EME | 4 modes / 69 cells | **0.3445 FlexCredit** | run record |
| coupler segment EME | 6 modes / 69 cells | **0.3783 FlexCredit** | **balance-closed** (dry-run = charge, to 1e-16) |
| all-device 3D FDTD | — | ≈ 30 FlexCredit | **not run** |

Costs are as recorded for each run. Only the 6-mode coupler run has a per-run balance-difference
closure; the others are the figures carried in their run records.

**Which script produces which table.** §1 (both taper readings) ← `sim/run_fdtd_short.py`. §2's column sums and the 4-mode conversion ← `sim/run_eme_coupler.py --modes 4` followed by `sim/analyze_coupler_eme.py`; §2's **6-mode** row ← the same run command with `--modes 6` followed by `sim/analyze_coupler_eme_modes.py --hdf5 <hdf5>` (mode-count parameterised). §2's **`passive`** row ← `sim/run_eme_passive.py --modes 2`, with the recorded analysis output in `notes/log_an_passive_m2.txt`.

Shipped with the repository: the validation data (`data/S_PARAMS.csv`, `data/PORT_MODE_NEFF.csv`,
`data/PORT_MODE_FIELDS.png`), the run records (`notes/`), and the figure used in §2
(`figures/colsum_and_conversion_vs_modes.png`, with its generating script in `tools/`).
Large outputs (cloud `.hdf5`, local solver projects) are not shipped; they are results rather than
dependencies and the scripts above regenerate them.


## 7. What this report does not solve

Recap of §4, in closing form:

- the cause of the boundary stated in §3 — **not identified**;
- a larger port-mode basis (10 or more), or a different constraint — **not tested**, outside the
  scope of this work;
- the output section (S-bend + `L_t`, x 405…537.8 µm) — **not simulated**;
- device-level return loss — **not addressed**.


---

### Files in the companion repository

| path | contents |
|---|---|
| `README.md` | the same report as a repository landing page, with the full scope statement |
| `psr_bilevel_eme_scope.ipynb` | this report as a notebook, mirrored here so the two copies cannot drift |
| `sim/` | the scripts reproduced above, plus `SIMULATION_SETUP.md` |
| `data/` | `S_PARAMS.csv`, `PORT_MODE_NEFF.csv`, `PORT_MODE_FIELDS.png` — the validation data |
| `notes/` | the run records (4-mode, 6-mode convergence) and the `passive` analysis log |
| `figures/` | the §2 figure and the model geometry figures, with a figure-provenance statement |
| `tools/` | the script that regenerates the §2 figure, and the figure-channel labelling module |

Repository: https://github.com/55093693k-dot/psr-bilevel-eme-scope

### Quoting rules

**Referenceable** — coupler segment `TE1 -> lower-arm TE0 = 99.233%` (6 modes, converged);
taper segment `TM0 -> TE1 = 98.35%` (3D FDTD).

**Not referenceable** — device-level absolute insertion loss / return loss; see §4–§5 for the basis.
